# Speculative decoding — GPU run

Runs the full characterization on a real CUDA GPU (vs the Apple-MPS results in the repo). The key payoff is the **serving roofline** (stage 7): on a real GPU the verify cost grows with batch size, so the speedup decays under load — the transition that MPS can't show.

**How to use:** Runtime → Change runtime type → GPU (T4 is free). Then Runtime → Run all. At the end it zips `data/` + `figures/` for download.

- Free **T4 (16GB)** → auto-uses the **3B** target (fits in memory, still shows the GPU rolloff).
- **L4/A100 (24GB+)** (Colab Pro) → uses the full **7B** target to match the repo's headline.

In [ ]:
!pip install -q -U transformers datasets accelerate
!git clone -q https://github.com/nazanindev/llm-speculative-decoding
%cd llm-speculative-decoding

In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), 'No GPU — set Runtime type to GPU'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
name = torch.cuda.get_device_name(0)
if mem >= 22:
    TARGET, DRAFT = '7B', '1.5B'
    os.environ['SPECDEC_TARGETS'] = '3B,7B'; os.environ['SPECDEC_DRAFTS'] = '0.5B,1.5B'
    os.environ['SPECDEC_MODELS'] = '0.5B,1.5B,3B,7B'
else:
    TARGET, DRAFT = '3B', '1.5B'
    os.environ['SPECDEC_TARGETS'] = '3B'; os.environ['SPECDEC_DRAFTS'] = '0.5B,1.5B'
    os.environ['SPECDEC_MODELS'] = '0.5B,1.5B,3B'
print(f'GPU: {name}  {mem:.0f} GB  ->  target {TARGET}, draft {DRAFT}')

In [ ]:
def run(cmd):
    print('\n$', cmd)
    p = subprocess.run(cmd, shell=True, env=os.environ, capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode: print('STDERR:', p.stderr[-2000:])

run('python scripts/01_acceptance.py')
run(f'python scripts/04_sweep.py {DRAFT} {TARGET}')
run(f'python scripts/05_sampled.py {DRAFT} {TARGET}')
run(f'python scripts/06_cross.py {TARGET} {DRAFT}')
run(f'python scripts/07_serving.py {DRAFT} {TARGET}')
run('python scripts/03_figures.py')

In [ ]:
# zip results for download (data/*.json + figures/*.png), then grab it
!zip -qr gpu_results.zip data figures
from google.colab import files
files.download('gpu_results.zip')